# SignBridge — Train ISL & BSL Models (Kaggle Notebook)

This notebook trains new sign-language recognition models for:
- 🇮🇳 **Indian Sign Language (ISL)** — two-handed fingerspelling alphabet
- 🇬🇧 **British Sign Language (BSL)** — two-handed fingerspelling alphabet

and exports them in exactly the file format SignBridge already expects.

**Important Setup:**
1. Go to https://www.kaggle.com/settings → API → Create New Token
2. In Kaggle Notebook, go to **Addons** tab (right panel) → **Secrets**
3. Click **+ Add a secret** and name it `kaggle_json`
4. Paste the entire content of your downloaded `kaggle.json` as the value
5. Toggle the switch to make it available to the notebook

**Runtime:** Free Kaggle CPU is enough — no GPU required.

In [ ]:
# ⚠️ CRITICAL: Set environment variables BEFORE importing TensorFlow
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['KERAS_BACKEND'] = 'tensorflow'

# Install dependencies
!pip -q install kaggle mediapipe==0.10.14 opencv-python-headless scikit-learn tensorflowjs==4.10.0 --upgrade

# Now import TensorFlow (will use legacy Keras)
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')), "(fine either way)")

In [ ]:
# 2. Kaggle API credentials from Secrets
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    kaggle_json_content = user_secrets.get_secret("kaggle_json")
    
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        f.write(kaggle_json_content)
    
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print("✅ Kaggle credentials installed successfully from Secrets.")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Please add a secret named 'kaggle_json' in Addons → Secrets panel.")
    raise

In [ ]:
# 3. Download datasets
os.makedirs('/kaggle/working/data/isl_raw', exist_ok=True)
os.makedirs('/kaggle/working/data/bsl_raw', exist_ok=True)

!kaggle datasets download -d eraakash/indian-sign-language-hand-landmarks-dataset -p /kaggle/working/data/isl_raw --unzip
!kaggle datasets download -d alifsathar/bsl-fingerspelling-dataset -p /kaggle/working/data/bsl_raw --unzip

print("\n--- ISL raw files ---")
for root, _, fs in os.walk('/kaggle/working/data/isl_raw'):
    for f in fs[:3]:
        print(os.path.join(root, f))

print("\n--- BSL folder structure ---")
count = 0
for root, dirs, fs in os.walk('/kaggle/working/data/bsl_raw'):
    print(root, '-> dirs:', dirs[:3], 'files:', fs[:2])
    count += 1
    if count > 5:
        break

In [ ]:
# 4. Shared feature pipeline — matches normalize.js exactly
import numpy as np

WRIST = 0
NUM_LANDMARKS = 21

def normalize_landmarks(pts):
    pts = np.asarray(pts, dtype=np.float32).reshape(NUM_LANDMARKS, 3)
    wrist = pts[WRIST].copy()
    centered = pts - wrist
    max_val = float(np.max(np.abs(centered))) if centered.size else 0.0
    scaled = centered / max_val if max_val > 0 else centered
    return scaled.reshape(-1).astype(np.float32)

def build_two_hand_126(hand0, hand1):
    a = normalize_landmarks(hand0) if hand0 is not None else np.zeros(63, dtype=np.float32)
    b = normalize_landmarks(hand1) if hand1 is not None else np.zeros(63, dtype=np.float32)
    return np.concatenate([a, b]).astype(np.float32)

NUM_FEATURES_TWO_HAND = 126
print("Feature pipeline ready. Output shape per sample:", NUM_FEATURES_TWO_HAND)

In [ ]:
# 5. Build ISL training set
import pandas as pd
import glob
import string

isl_csvs = glob.glob('/kaggle/working/data/isl_raw/**/*.csv', recursive=True)
assert isl_csvs, "No CSV found under /kaggle/working/data/isl_raw"

isl_df = pd.concat([pd.read_csv(f) for f in isl_csvs], ignore_index=True)
print("Combined ISL shape:", isl_df.shape)

# Find label column
def find_label_column(df):
    for cand in ['label', 'class', 'letter', 'sign', 'target', 'Label', 'Class']:
        if cand in df.columns:
            return cand
    non_numeric = [c for c in df.columns if df[c].dtype == object]
    if non_numeric:
        return non_numeric[-1]
    raise ValueError("Couldn't auto-detect label column")

LABEL_COL = find_label_column(isl_df)
print("Detected label column:", LABEL_COL)

# Find coordinate columns (exclude metadata)
exclude_keywords = ['uses_two_hands', 'use_two_hands', 'two_hands', 'is_two_handed', 'id', 'name', 'file', 'path', 'index']
coord_cols = [c for c in isl_df.columns 
              if c != LABEL_COL 
              and pd.api.types.is_numeric_dtype(isl_df[c]) 
              and not any(keyword in c.lower() for keyword in exclude_keywords)]

n = len(coord_cols)
print(f"Detected {n} numeric coordinate columns")

if n == 63:
    HANDS_PER_ROW = 1
elif n == 126:
    HANDS_PER_ROW = 2
else:
    raise ValueError(f"Unexpected coordinate-column count ({n}), expected 63 or 126")

# Build dataset
def row_to_hand(flat63):
    return np.asarray(flat63, dtype=np.float32).reshape(21, 3)

coords = isl_df[coord_cols].to_numpy(dtype=np.float32)
labels = isl_df[LABEL_COL].apply(lambda x: chr(ord('A') + int(x))).to_numpy()

X_isl, y_isl = [], []

for i in range(len(isl_df)):
    if HANDS_PER_ROW == 1:
        hand0, hand1 = row_to_hand(coords[i]), None
    else:
        h0 = row_to_hand(coords[i, :63])
        h1 = row_to_hand(coords[i, 63:126])
        hand1 = None if np.allclose(h1, 0) else h1
        hand0 = h0
        if np.allclose(hand0, 0) and hand1 is not None:
            hand0, hand1 = hand1, None

    X_isl.append(build_two_hand_126(hand0, hand1))
    y_isl.append(labels[i])

    if hand0 is not None and hand1 is not None:
        X_isl.append(build_two_hand_126(hand1, hand0))
        y_isl.append(labels[i])

X_isl = np.stack(X_isl)
y_isl = np.array(y_isl)

# Keep only A-Z letters
mask = np.array([lbl in set(string.ascii_uppercase) for lbl in y_isl])
X_isl, y_isl = X_isl[mask], y_isl[mask]

print("ISL dataset ready:", X_isl.shape, y_isl.shape)
print("Classes:", sorted(set(y_isl)))

In [ ]:
# 6. Build BSL training set from images
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
hands_detector = mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.5)

bsl_images = []
bsl_labels = []

for letter in string.ascii_uppercase:
    letter_dir = f'/kaggle/working/data/bsl_raw/train/{letter}'
    if not os.path.exists(letter_dir):
        continue
    for fname in os.listdir(letter_dir):
        if fname.endswith(('.jpg', '.jpeg', '.png')):
            bsl_images.append(os.path.join(letter_dir, fname))
            bsl_labels.append(letter)

print(f"Found {len(bsl_images)} BSL images")
print("Labels detected:", sorted(set(bsl_labels)))

X_bsl, y_bsl = [], []
skipped = 0

for i, (img_path, label) in enumerate(zip(bsl_images, bsl_labels)):
    if i % 2000 == 0:
        print(f"  processed {i}/{len(bsl_images)} images — {len(X_bsl)} samples so far, {skipped} skipped")
    
    img = cv2.imread(img_path)
    if img is None:
        skipped += 1
        continue
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = hands_detector.process(img_rgb)
    
    if result.multi_hand_landmarks is None:
        skipped += 1
        continue
    
    hands = result.multi_hand_landmarks
    hand0 = np.array([[lm.x, lm.y, lm.z] for lm in hands[0].landmark]) if len(hands) > 0 else None
    hand1 = np.array([[lm.x, lm.y, lm.z] for lm in hands[1].landmark]) if len(hands) > 1 else None
    
    X_bsl.append(build_two_hand_126(hand0, hand1))
    y_bsl.append(label)
    
    if hand0 is not None and hand1 is not None:
        X_bsl.append(build_two_hand_126(hand1, hand0))
        y_bsl.append(label)

hands_detector.close()

X_bsl = np.stack(X_bsl)
y_bsl = np.array(y_bsl)

print(f"\nDone. {len(X_bsl)} samples built, {skipped} images skipped.")
print("Classes:", sorted(set(y_bsl)))

In [ ]:
# 7. Shared model architecture + training function
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def build_model(num_features, num_classes):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(num_features,)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(num_classes, activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def train_language_model(X, y, lang_name, epochs=60, batch_size=64):
    print(f"\n{'='*60}\nTraining {lang_name.upper()}\n{'='*60}")
    
    encoder = LabelEncoder()
    y_int = encoder.fit_transform(y)
    class_names = list(encoder.classes_)
    num_classes = len(class_names)
    print(f"{num_classes} classes: {class_names}")
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y_int, test_size=0.15, random_state=42, stratify=y_int
    )
    
    model = build_model(X.shape[1], num_classes)
    model.summary()
    
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4)
    
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop, reduce_lr],
        verbose=2,
    )
    
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"\n{lang_name.upper()} validation accuracy: {val_acc:.4f}")
    
    return model, encoder, class_names, val_acc

In [ ]:
# 8. Train ISL
isl_model, isl_encoder, isl_classes, isl_val_acc = train_language_model(X_isl, y_isl, "isl")

In [ ]:
# 9. Train BSL
bsl_model, bsl_encoder, bsl_classes, bsl_val_acc = train_language_model(X_bsl, y_bsl, "bsl")

In [ ]:
# 10. Export to TF.js
import subprocess
import json
import datetime
import shutil

def export_language(model, class_names, val_acc, lang_id, out_root='/kaggle/working/export'):
    out_dir = f'{out_root}/{lang_id}'
    saved_model_dir = f'/kaggle/working/temp_saved_model_{lang_id}'
    temp_h5_path = f'/kaggle/working/temp_model_{lang_id}.h5'
    
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(saved_model_dir, exist_ok=True)
    
    print(f"\n--- Exporting {lang_id.upper()} ---")
    
    # Save as .h5 first (legacy Keras format)
    model.save(temp_h5_path)
    print(f"Saved .h5 file to {temp_h5_path}")
    
    # Reload and save as SavedModel
    loaded_model = tf.keras.models.load_model(temp_h5_path)
    tf.saved_model.save(loaded_model, saved_model_dir)
    print(f"Saved SavedModel to {saved_model_dir}")
    
    # Cleanup temp h5
    if os.path.exists(temp_h5_path):
        os.remove(temp_h5_path)
    
    # Convert to TF.js
    cmd = [
        'tensorflowjs_converter',
        '--input_format=tf_saved_model',
        '--output_format=tfjs_layers_model',
        '--output_json_max_inlined_tensor_size=1048576',
        saved_model_dir,
        out_dir
    ]
    
    print(f"Running converter: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print("❌ Conversion failed:")
        print(result.stderr)
        raise RuntimeError("TensorFlow.js conversion failed")
    
    print("✅ Conversion successful.")
    
    # Generate label_encoder.json
    encoder_data = {"classes": class_names}
    with open(f'{out_dir}/label_encoder.json', 'w') as f:
        json.dump(encoder_data, f)
    print("Created label_encoder.json")
    
    # Generate model_metadata.json
    metadata = {
        "num_features": 126,
        "num_classes": len(class_names),
        "class_names": class_names,
        "test_accuracy": float(val_acc),
        "pipeline": "two-hand-126",
        "trained_at": datetime.datetime.now().isoformat(),
        "framework": "tensorflowjs",
        "version": "1.0.0"
    }
    with open(f'{out_dir}/model_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    print("Created model_metadata.json")
    
    # Cleanup SavedModel dir
    if os.path.exists(saved_model_dir):
        shutil.rmtree(saved_model_dir)
    
    return out_dir

# Run exports
isl_out = export_language(isl_model, isl_classes, isl_val_acc, 'isl')
bsl_out = export_language(bsl_model, bsl_classes, bsl_val_acc, 'bsl')
print("\n🎉 All models exported successfully!")

In [ ]:
# 11. Validate exports
def validate_export(out_dir, expected_features, expected_classes):
    with open(f'{out_dir}/model.json') as f:
        model_json = json.load(f)
    assert 'modelTopology' in model_json, f"{out_dir}/model.json missing modelTopology"
    
    with open(f'{out_dir}/label_encoder.json') as f:
        enc = json.load(f)
    assert len(enc['classes']) == expected_classes, "label_encoder.json class count mismatch"
    
    with open(f'{out_dir}/model_metadata.json') as f:
        meta = json.load(f)
    assert meta['num_features'] == expected_features, "model_metadata.json feature count mismatch"
    
    print(f"✅ {out_dir} passes sanity checks — {len(enc['classes'])} classes, {meta['num_features']} features")

validate_export(isl_out, 126, len(isl_classes))
validate_export(bsl_out, 126, len(bsl_classes))

print(f"\nISL validation accuracy: {isl_val_acc:.4f}")
print(f"BSL validation accuracy: {bsl_val_acc:.4f}")

In [ ]:
# 12. Package for download
shutil.make_archive('/kaggle/working/signbridge_isl_bsl_models', 'zip', '/kaggle/working/export')
print("Created /kaggle/working/signbridge_isl_bsl_models.zip")
print("\nDownload from the Output tab on the right panel!")